# Basic Pitch Onset-Threshold Sweep — tuned by tab accuracy

Standalone companion to **`AudioToTabCAGED (2).ipynb`**. It sweeps the Basic Pitch `onset_threshold` (and `frame_threshold`) that your pipeline currently leaves at the library default, selects the best value **on the validation split by `exact_tab_f1`**, then reports on the test split **once**.

### Why this is worth doing
Your `run_basic_pitch_notes` calls `basic_pitch_predict(audio_path)` with Basic Pitch's *default* onset/frame thresholds, then post-filters notes by their `amplitude` field. Your team tuned that **amplitude** post-filter (0.30 → 0.40). The **onset_threshold inside `predict()`** is a separate, never-swept knob — a threshold sweep on a held-out player showed note-F1 climbing meaningfully from the 0.40-equivalent operating point up toward ~0.60-0.65. This notebook checks whether that helps the thing you actually care about: **end-to-end tab F1**, not note F1.

### How it plugs in
It runs your real notebook with `%run` to import every function unchanged (`evaluate_audio_to_tab_record`, `AUDIO_ASSIGNMENT_METHODS`, `VAL_RECORDS`, `TEST_RECORDS`, `find_audio_for_record`, `basic_pitch_predict`, …), then **monkey-patches only `run_basic_pitch_notes`** to (a) pass `onset_threshold`/`frame_threshold` into `basic_pitch_predict` and (b) use a **threshold-aware cache key** so the sweep never serves stale notes from a different threshold. Nothing in your algorithm changes.

### Discipline baked in
- Tune on `VAL_RECORDS`, report on `TEST_RECORDS` exactly once.
- Select by `exact_tab_f1` (end-to-end). `tab_accuracy_given_pitch_match` (TDR) is assignment-only and barely moves with detection threshold, so it is the wrong selection target — shown for context only.
- Playability columns are printed at the winning threshold so a precision shift that quietly worsens fret jumps doesn't slip through.


## 1. Mount Drive + point to your notebook

In [3]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# EDIT if your path differs. This is the self-contained algorithm notebook with chord voicings.
PIPELINE_NB = '/content/drive/MyDrive/Capstone/Code/AudioToTabCAGED.ipynb'
assert Path(PIPELINE_NB).exists(), f"Not found: {PIPELINE_NB}  — fix PIPELINE_NB to point at your notebook."
print("Pipeline notebook:", PIPELINE_NB)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pipeline notebook: /content/drive/MyDrive/Capstone/Code/AudioToTabCAGED.ipynb


## 2. Run your pipeline to import everything

`%run` executes the notebook in this namespace, so all functions, configs, and the `TRAIN/VAL/TEST` split land here exactly as defined there. This runs its end-to-end TEST eval too (cell 46) — a few minutes; harmless, it just warms the note cache. Set `MAX_AUDIO_RECORDINGS` small in your config first if you want it faster.

In [4]:
%run "$PIPELINE_NB"

# Sanity-check the symbols we depend on all exist.
_needed = ['run_basic_pitch_notes','basic_pitch_predict','evaluate_audio_to_tab_record',
           'AUDIO_ASSIGNMENT_METHODS','VAL_RECORDS','TEST_RECORDS','find_audio_for_record',
           'BASIC_PITCH_CACHE_DIR','BASIC_PITCH_AMPLITUDE_THRESHOLD','BASIC_PITCH_MIN_MIDI',
           'BASIC_PITCH_MAX_MIDI','make_audio_record_from_gt']
_missing = [n for n in _needed if n not in dir()]
assert not _missing, f"Missing symbols after %run: {_missing}"
print("All required symbols present. VAL:", len(VAL_RECORDS), "TEST:", len(TEST_RECORDS))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.
OUTPUT_DIR: /content/drive/MyDrive/Capstone/outputs/fretboard_playability
OUTPUT_DIR exists: True

Data root candidates visible to this runtime:
 - /content/drive/MyDrive/Capstone/FullGuitarSetData | exists: True
 - /content/drive/MyDrive/FullGuitarSetData | exists: False
 - /content/drive/MyDrive/Capstone/GuitarSet | exists: True
 - /content/drive/MyDrive/GuitarSet | exists: False
 - /content/drive/MyDrive/Capstone | exists: True
 - /content/drive/MyDrive | exists: True
 - /mnt/data/fretwork_repo/GuitarSet | exists: False
 - /mnt/data/fretwork_repo | exists: False

Top-level MyDrive folders/files visible to Colab:
 - labels.csv
 - yelp_dataset.tar
 - Spring 2025
 - Tell a compelling story. Example. Fall 2020. Airline Pricing.gdoc
 - DATASCI200 Project Proposal.gdoc
 - personalized bus routes.fall 2024.gdoc
 - Final Report T

,string,string_name,fret,midi,pitch_class
0,0,low_E,0,40,4
1,0,low_E,1,41,5
2,0,low_E,2,42,6
3,0,low_E,3,43,7
4,0,low_E,4,44,8
5,0,low_E,5,45,9
6,0,low_E,6,46,10
7,0,low_E,7,47,11
8,0,low_E,8,48,0
9,0,low_E,9,49,1


Example positions for MIDI 64 / E4:


,string,string_name,fret,midi,pitch_class
0,0,low_E,24,64,4
1,1,A,19,64,4
2,2,D,14,64,4
3,3,G,9,64,4
4,4,B,5,64,4
5,5,high_E,0,64,4


,key,root,root_pc,mode,scale_pcs,scale_notes,diatonic_chords
0,C major,C,0,major,"[0, 2, 4, 5, 7, 9, 11]","[C, D, E, F, G, A, B]","[{'degree': 1, 'root_pc': 0, 'root': 'C', 'qua..."
1,C minor,C,0,minor,"[0, 2, 3, 5, 7, 8, 10]","[C, D, D#, F, G, G#, A#]","[{'degree': 1, 'root_pc': 0, 'root': 'C', 'qua..."
2,C# major,C#,1,major,"[1, 3, 5, 6, 8, 10, 0]","[C#, D#, F, F#, G#, A#, C]","[{'degree': 1, 'root_pc': 1, 'root': 'C#', 'qu..."
3,C# minor,C#,1,minor,"[1, 3, 4, 6, 8, 9, 11]","[C#, D#, E, F#, G#, A, B]","[{'degree': 1, 'root_pc': 1, 'root': 'C#', 'qu..."
4,D major,D,2,major,"[2, 4, 6, 7, 9, 11, 1]","[D, E, F#, G, A, B, C#]","[{'degree': 1, 'root_pc': 2, 'root': 'D', 'qua..."


...


REAL AUDIO → BASIC PITCH → ALGORITHM TAB
REAL AUDIO PREDICTED TAB — 05_SS2-88-F_solo — combined_all_tuned
----------------------------------------------------------------
e|---------6--8--9--6-----|------------------------|------------------------|------------------------|
B|---------6-----------6--|---------6--------------|------------9--11-9--8--|6-----------------6--9--|
G|------------------------|------------8-----------|---------8--------------|------------------------|
D|------------------------|------------------------|------------------------|------------------------|
A|------------------------|------------------------|------------------------|------------------------|
E|------------------------|------------------------|------------------------|------------------------|

e|------------------------|------------------------|------------------------|------------------5--8--|
B|8--------------8--------|------------------8--6--|6-----------------6-----|------------------6-----|
G|-

,method,recordings,n_gt_notes,n_basic_pitch_notes,pitch_precision,pitch_recall,pitch_f1,exact_tab_precision,exact_tab_recall,exact_tab_f1,tab_accuracy_given_pitch_match,string_accuracy_given_pitch_match,fret_accuracy_given_pitch_match,mean_runtime_sec_per_recording
1,caged_voiced,54,10207,9561,0.802,0.7512,0.7758,0.5549,0.5197,0.5367,0.6918,0.6918,0.6918,0.7057
0,caged_box,54,10207,9561,0.802,0.7512,0.7758,0.5497,0.5149,0.5318,0.6854,0.6854,0.6854,1.8391
2,combined_all_tuned,54,10207,9561,0.802,0.7512,0.7758,0.5191,0.4862,0.5021,0.6472,0.6472,0.6472,0.7478



Saved report table to: /content/drive/MyDrive/Capstone/outputs/audio_to_tab_basic_pitch_heldout/audio_to_tab_report_table_heldout_test.csv

Interpretation:
The previous held-out result measured fretboard assignment using ground-truth GuitarSet notes.
This result starts from raw audio, runs Basic Pitch first, then predicts string/fret positions.
So exact_tab_f1 is the stricter end-to-end audio-to-tab metric.
Matched notes (detected + pitch-correct): 7668
  exact-correct placement : 5256  (68.5%)
  wrong placement         : 2412  (31.5%)

Of the WRONG notes — how many strings off:
  |Δstring|=1:  2309  (95.7%)  ADJACENT (one string off)
  |Δstring|=2:    88  ( 3.6%)  two strings off
  |Δstring|=3:    14  ( 0.6%)  three strings off
  |Δstring|=5:     1  ( 0.0%)  5 off

>>> 95.7% of wrong notes are exactly ONE string off (adjacent).
>>> That's 30.1% of all matched notes (the rest place correctly or miss by 2+ strings).

Fret-diff distribution (sanity: adjacent swaps cluster at ±4/±5):
fre

slice,ALL,comp,solo
method,,,
caged_box,0.6854,0.7199,0.6105
caged_voiced,0.6918,0.7308,0.6072
combined_all_tuned,0.6472,0.6677,0.6026


All required symbols present. VAL: 54 TEST: 54


## 3. Monkey-patch `run_basic_pitch_notes` (threshold-aware + threshold-keyed cache)

Identical body to yours, with two changes: it reads module-level `BP_ONSET_THRESHOLD` / `BP_FRAME_THRESHOLD` and passes them to `basic_pitch_predict`, and the cache filename encodes the thresholds so each operating point caches separately. **The threshold-aware cache key is the critical bit** — your original key is filename-only, so without this every sweep step after the first would silently reload the first threshold's notes.

In [5]:
import pandas as pd, numpy as np

# Globals the patched runner reads. The sweep sets these before each eval pass.
BP_ONSET_THRESHOLD = 0.5   # Basic Pitch library default
BP_FRAME_THRESHOLD = 0.3

def _basic_pitch_cache_path_thr(audio_path, onset_threshold, frame_threshold):
    safe = Path(audio_path).stem.replace('/', '_')
    return BASIC_PITCH_CACHE_DIR / f'{safe}_on{onset_threshold:.2f}_fr{frame_threshold:.2f}_bp_notes.csv'

def run_basic_pitch_notes(audio_path,
                          amplitude_threshold=BASIC_PITCH_AMPLITUDE_THRESHOLD,
                          min_midi=BASIC_PITCH_MIN_MIDI,
                          max_midi=BASIC_PITCH_MAX_MIDI,
                          use_cache=True,
                          onset_threshold=None,
                          frame_threshold=None):
    """Patched: threshold-aware Basic Pitch + threshold-keyed cache. Falls back to BP_* globals."""
    if onset_threshold is None: onset_threshold = BP_ONSET_THRESHOLD
    if frame_threshold is None: frame_threshold = BP_FRAME_THRESHOLD
    audio_path = Path(audio_path)
    cache_path = _basic_pitch_cache_path_thr(audio_path, onset_threshold, frame_threshold)

    if use_cache and cache_path.exists():
        return pd.read_csv(cache_path).to_dict('records')

    _, _, note_events = basic_pitch_predict(str(audio_path),
                                            onset_threshold=onset_threshold,
                                            frame_threshold=frame_threshold)
    notes = []
    for event in note_events:
        start, end, pitch_midi, amplitude = event[0], event[1], event[2], event[3]
        if float(amplitude) < amplitude_threshold:
            continue
        midi = int(round(float(pitch_midi)))
        if midi < min_midi or midi > max_midi:
            continue
        notes.append({
            'start': float(start),
            'duration': float(end - start),
            'midi': midi,
            'pitch_class': midi % 12,
            'note_name': midi_to_note_name_simple(midi),
            'amplitude': float(amplitude),
            'true_string': None,
            'true_fret': None,
            'source': 'basic_pitch',
        })
    notes = sorted(notes, key=lambda n: (n['start'], n['midi']))
    pd.DataFrame(notes).to_csv(cache_path, index=False)
    return notes

# Rebind the name in the module namespace so evaluate_audio_to_tab_record() calls the patched version.
import sys
setattr(sys.modules['__main__'], 'run_basic_pitch_notes', run_basic_pitch_notes)
print("Patched run_basic_pitch_notes (threshold-aware + threshold-keyed cache).")

Patched run_basic_pitch_notes (threshold-aware + threshold-keyed cache).


## 4. Pair validation + test records with audio

Mirrors how your cell 35 builds `paired_records` over `TEST_RECORDS`, applied to both splits.

In [6]:
def pair_records(records):
    pairs, missing = [], []
    for r in records:
        ap = find_audio_for_record(r)
        (missing if ap is None else pairs).append(r['recording'] if ap is None else (r, ap))
    return pairs, missing

paired_val, miss_val = pair_records(VAL_RECORDS)
paired_test, miss_test = pair_records(TEST_RECORDS)
print(f"Validation paired: {len(paired_val)}/{len(VAL_RECORDS)}  (missing {len(miss_val)})")
print(f"Test paired:       {len(paired_test)}/{len(TEST_RECORDS)}  (missing {len(miss_test)})")
if miss_val[:5]: print("  e.g. missing val:", miss_val[:5])

Validation paired: 54/54  (missing 0)
Test paired:       54/54  (missing 0)


## 5. Sweep helper

For a given `(onset, frame)`, sets the globals and runs your unchanged `evaluate_audio_to_tab_record` over the paired records for one method. Aggregates `exact_tab_f1` **pooled** (summed TP / summed predicted / summed GT) — the same way your `audio_summary_df` does — so numbers match your pipeline. TDR and playability proxies are averaged for context.

In [7]:
def eval_pairs_at_threshold(paired, onset_t, frame_t, method, label=''):
    global BP_ONSET_THRESHOLD, BP_FRAME_THRESHOLD
    BP_ONSET_THRESHOLD, BP_FRAME_THRESHOLD = onset_t, frame_t
    assign_fn = AUDIO_ASSIGNMENT_METHODS[method]
    rows, all_matches = [], []
    for record, audio_path in paired:
        metrics, _assigned, matches_df = evaluate_audio_to_tab_record(record, audio_path, assign_fn, method)
        rows.append(metrics)
        if matches_df is not None and len(matches_df):
            all_matches.append(matches_df)
    df = pd.DataFrame(rows)
    tp  = df['exact_tab_tp'].sum()
    npr = df['n_basic_pitch_notes'].sum()
    ngt = df['n_gt_notes'].sum()
    P = tp/npr if npr else 0.0
    R = tp/ngt if ngt else 0.0
    f1 = 2*P*R/(P+R) if (P+R) else 0.0
    # TDR weighted by matched-note count (matches your summary's _safe_weighted_avg)
    w = df['n_pitch_onset_matches'].fillna(0).astype(float)
    tdr = float(np.average(df['tab_accuracy_given_pitch_match'].fillna(0), weights=w)) if w.sum() else float('nan')
    pitch_recall = df['pitch_recall'].mean()
    out = {'onset': onset_t, 'frame': frame_t, 'method': method,
           'exact_tab_f1': f1, 'exact_tab_precision': P, 'exact_tab_recall': R,
           'TDR': tdr, 'pitch_recall_mean': pitch_recall,
           'n_pred': int(npr), 'n_gt': int(ngt), 'n_match': int(df['n_pitch_onset_matches'].sum())}
    if label:
        print(f"  [{label}] onset={onset_t:.2f} frame={frame_t:.2f}  "
              f"exact_tab_f1={f1:.4f}  P={P:.3f} R={R:.3f}  TDR={tdr:.4f}")
    matches_cat = pd.concat(all_matches, ignore_index=True) if all_matches else pd.DataFrame()
    return out, matches_cat

## 6. Sweep on VALIDATION → pick by `exact_tab_f1`

First pass runs `predict()` at each threshold, so it is the slow part (~15-25 min for 8 thresholds × validation set). The threshold-keyed cache makes any re-run instant. `caged_voiced` is the default method; change `METHOD` to sweep a different one.

In [8]:
METHOD = 'caged_voiced'                       # 'caged_box' | 'caged_voiced' | 'combined_all_tuned'
ONSET_GRID = [0.30, 0.40, 0.50, 0.60, 0.65, 0.70, 0.75, 0.80]
FRAME_T = 0.30                                # held fixed; see optional frame sweep below

print(f"Sweeping onset on VALIDATION  |  method={METHOD}  |  {len(paired_val)} recordings")
val_rows = []
for ot in ONSET_GRID:
    res, _ = eval_pairs_at_threshold(paired_val, ot, FRAME_T, METHOD, label='VAL')
    val_rows.append(res)

val_df = pd.DataFrame(val_rows)
best = val_df.loc[val_df['exact_tab_f1'].idxmax()]
best_onset = float(best['onset'])
print("\nValidation sweep:")
display(val_df[['onset','frame','exact_tab_f1','exact_tab_precision','exact_tab_recall','TDR']].round(4))
print(f"\n>>> Best onset on validation by exact_tab_f1: {best_onset:.2f} "
      f"(val exact_tab_f1={best['exact_tab_f1']:.4f}, default 0.50 baseline below)")

Sweeping onset on VALIDATION  |  method=caged_voiced  |  54 recordings
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/02_Funk3-98-A_solo_mic.wav...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_Jazz2-187-F#_comp_mic.wav...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_BN3-154-E_comp_mic.wav...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/01_Rock1-90-C#_solo_mic.wav...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/01_BN2-166-Ab_comp_mic.wav...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/05_Rock1-130-A_solo_mic.wav...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_Rock1-130-A_comp_mic.wav...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/04_Funk3-112-C#_comp_mic.wav...
Predicting MIDI for /content/drive/

,onset,frame,exact_tab_f1,exact_tab_precision,exact_tab_recall,TDR
0,0.30,0.3,0.4350,0.3879,0.4951,0.6757
1,0.40,0.3,0.4731,0.4448,0.5052,0.6728
2,0.50,0.3,0.4885,0.4813,0.4958,0.6656
3,0.60,0.3,0.4854,0.5017,0.4701,0.6528
4,0.65,0.3,0.4843,0.5142,0.4577,0.6510
5,0.70,0.3,0.4807,0.5243,0.4438,0.6504
6,0.75,0.3,0.4692,0.5255,0.4237,0.6434
7,0.80,0.3,0.4512,0.5217,0.3974,0.6372



>>> Best onset on validation by exact_tab_f1: 0.50 (val exact_tab_f1=0.4885, default 0.50 baseline below)


## 7. Report on TEST once — best vs default

In [9]:
res_best, m_best = eval_pairs_at_threshold(paired_test, best_onset, FRAME_T, METHOD)
res_def,  m_def  = eval_pairs_at_threshold(paired_test, 0.50,       FRAME_T, METHOD)

cmp = pd.DataFrame([{'config': f'tuned onset={best_onset:.2f}', **res_best},
                    {'config': 'default onset=0.50',           **res_def}])
print(f"TEST results — method={METHOD}")
display(cmp[['config','exact_tab_f1','exact_tab_precision','exact_tab_recall','TDR','n_pred','n_match']].round(4))

delta = res_best['exact_tab_f1'] - res_def['exact_tab_f1']
print(f"\nDelta exact_tab_f1 (tuned - default): {delta:+.4f}")
if delta > 0.003:
    print("Tuning the onset threshold gives a real end-to-end tab improvement. Bank it.")
elif delta < -0.003:
    print("Default is better here; the validation pick didn't transfer. Keep onset=0.50.")
else:
    print("Flat. Your operating point was already well-chosen — a clean, reportable null.")

Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/02_BN2-131-B_solo_mic.wav...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_Rock2-142-D_comp_mic.wav...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/05_Rock2-85-F_solo_mic.wav...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/04_Jazz3-137-Eb_comp_mic.wav...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/05_SS2-88-F_solo_mic.wav...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/01_SS1-100-C#_comp_mic.wav...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/04_Funk2-119-G_solo_mic.wav...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_SS2-107-Ab_solo_mic.wav...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/01_BN2-131-B_comp_mic.wav...
P

,config,exact_tab_f1,exact_tab_precision,exact_tab_recall,TDR,n_pred,n_match
0,tuned onset=0.50,0.5372,0.5553,0.5201,0.6924,9560,7668
1,default onset=0.50,0.5372,0.5553,0.5201,0.6924,9560,7668



Delta exact_tab_f1 (tuned - default): +0.0000
Flat. Your operating point was already well-chosen — a clean, reportable null.


## 8. Playability check at the winning threshold

A higher onset threshold feeds fewer, cleaner notes into the CAGED layer. Confirm that didn't worsen the hand-motion profile even if tab F1 rose. Compares fret-jump proxies on the matched notes, tuned vs default.

In [10]:
def playability_proxies(matches_df):
    if matches_df is None or matches_df.empty or 'pred_fret' not in matches_df:
        return {}
    d = matches_df.dropna(subset=['pred_fret']).copy()
    d = d.sort_values(['recording','pred_start'])
    jumps = d.groupby('recording')['pred_fret'].apply(lambda s: s.astype(float).diff().abs())
    jumps = jumps.dropna()
    return {'n_matched_notes': int(len(d)),
            'avg_fret_jump': float(jumps.mean()) if len(jumps) else float('nan'),
            'large_jump_rate(>4)': float((jumps > 4).mean()) if len(jumps) else float('nan'),
            'mean_pred_fret': float(d['pred_fret'].astype(float).mean())}

play = pd.DataFrame([{'config': f'tuned onset={best_onset:.2f}', **playability_proxies(m_best)},
                     {'config': 'default onset=0.50',            **playability_proxies(m_def)}])
print("Playability proxies on matched notes (lower jump = stiller hand):")
display(play.round(4))
print("\nIf avg_fret_jump / large_jump_rate are flat or lower at the tuned threshold, the F1 gain is clean.")

Playability proxies on matched notes (lower jump = stiller hand):


,config,n_matched_notes,avg_fret_jump,large_jump_rate(>4),mean_pred_fret
0,tuned onset=0.50,7668,1.5201,0.0503,4.9791
1,default onset=0.50,7668,1.5201,0.0503,4.9791



If avg_fret_jump / large_jump_rate are flat or lower at the tuned threshold, the F1 gain is clean.


## 9. (Optional) Joint onset × frame sweep on validation

Frame threshold controls note sustain/segmentation; usually secondary to onset, but cheap to check once the cache is warm for the onset values above. Uncomment to run.

In [ ]:
# FRAME_GRID = [0.20, 0.30, 0.40]
# joint = []
# for ot in ONSET_GRID:
#     for ft in FRAME_GRID:
#         res, _ = eval_pairs_at_threshold(paired_val, ot, ft, METHOD)
#         joint.append(res)
# joint_df = pd.DataFrame(joint).sort_values('exact_tab_f1', ascending=False)
# display(joint_df[['onset','frame','exact_tab_f1','exact_tab_precision','exact_tab_recall','TDR']].round(4).head(12))
# bj = joint_df.iloc[0]
# print(f"Best (onset,frame) on val: ({bj['onset']:.2f},{bj['frame']:.2f})  exact_tab_f1={bj['exact_tab_f1']:.4f}")
# # Then re-run Section 7 with FRAME_T = bj['frame'] and best_onset = bj['onset'] to confirm on TEST once.